In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping

# Load dataset
# The dataset contains Twitter text related to Apple and Google
df = pd.read_csv('/content/judge-1377884607_tweet_product_company.csv', encoding='latin1')

# Drop the 'emotion_in_tweet_is_directed_at' column as requested
df = df.drop(columns=['emotion_in_tweet_is_directed_at'], errors='ignore')

# Rename columns for easier access
df.columns = ['text', 'sentiment']
df = df.dropna(subset=['text', 'sentiment'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9092 entries, 0 to 9092
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       9092 non-null   object
 1   sentiment  9092 non-null   object
dtypes: object(2)
memory usage: 213.1+ KB


In [ ]:
df.head()

,text,sentiment
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Positive emotion


text processing

In [ ]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Remove URLs
    text = re.sub(r'\@\w+|\#','', text) # Remove mentions and hashtags
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation
    text = " ".join([word for word in text.split() if word not in stop_words])
    return text

df['text'] = df['text'].apply(clean_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
df.head()

,text,sentiment
0,3g iphone 3 hrs tweeting rise_austin dead need...,Negative emotion
1,know awesome ipadiphone app youll likely appre...,Positive emotion
2,wait ipad 2 also sale sxsw,Positive emotion
3,hope years festival isnt crashy years iphone a...,Negative emotion
4,great stuff fri sxsw marissa mayer google tim ...,Positive emotion


3. Label Encoding and Tokenization
The labels ('positive', 'negative', 'neutral', 'no_idea') must be converted into numerical format.

In [ ]:
# Encode the four sentiment classes [cite: 3, 4]
le = LabelEncoder()
df['sentiment'] = le.fit_transform(df['sentiment'])
num_classes = len(le.classes_)

# Tokenization
max_features = 5000
tokenizer = Tokenizer(num_words=max_features, split=' ')
tokenizer.fit_on_texts(df['text'].values)
X = tokenizer.texts_to_sequences(df['text'].values)
X = pad_sequences(X) # Ensure all sequences have the same length

Y = pd.get_dummies(df['sentiment']).values
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)


In [ ]:
df

,text,sentiment
0,3g iphone 3 hrs tweeting rise_austin dead need...,1
1,know awesome ipadiphone app youll likely appre...,3
2,wait ipad 2 also sale sxsw,3
3,hope years festival isnt crashy years iphone a...,1
4,great stuff fri sxsw marissa mayer google tim ...,3
...,...,...
9088,ipad everywhere sxsw link,3
9089,wave buzz rt interrupt regularly scheduled sxs...,2
9090,googles zeiger physician never reported potent...,2
9091,verizon iphone customers complained time fell ...,2


4. Build LSTM Model

In [ ]:
from tensorflow.keras.layers import Input

# Define parameters
max_features = 5000
embed_dim = 128
lstm_out = 196
input_length = X.shape[1] # The length of your padded sequences

model = Sequential()

model.add(Input(shape=(input_length,)))
model.add(Embedding(max_features, embed_dim))
model.add(SpatialDropout1D(0.4))
model.add(LSTM(lstm_out, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(4, activation='softmax')) # 4 classes: pos, neg, neu, no_idea [cite: 3]

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Now the summary will show the parameters correctly
print(model.summary())

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 24, 128)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ (None, 24, 128)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 196)            │       254,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           788 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 895,588 (3.42 MB)

 Trainable params: 895,588 (3.42 MB)

 Non-trainable params: 0 (0.00 B)

None


5. Model Training

In [ ]:
epochs = 10
batch_size = 32

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(X_train, Y_train,
                    epochs=epochs,
                    batch_size=batch_size,
                    validation_data=(X_test, Y_test),
                    callbacks=[early_stop],
                    verbose=2)

Epoch 1/10
228/228 - 39s - 171ms/step - accuracy: 0.6028 - loss: 0.9081 - val_accuracy: 0.6278 - val_loss: 0.8664
Epoch 2/10
228/228 - 42s - 184ms/step - accuracy: 0.6921 - loss: 0.7365 - val_accuracy: 0.6504 - val_loss: 0.8030
Epoch 3/10
228/228 - 35s - 153ms/step - accuracy: 0.7597 - loss: 0.6036 - val_accuracy: 0.6476 - val_loss: 0.8464
Epoch 4/10
228/228 - 40s - 177ms/step - accuracy: 0.8009 - loss: 0.5185 - val_accuracy: 0.6476 - val_loss: 0.8824
Epoch 5/10
228/228 - 41s - 179ms/step - accuracy: 0.8159 - loss: 0.4633 - val_accuracy: 0.6586 - val_loss: 0.9311


6. Evaluate Performance

In [ ]:
score, acc = model.evaluate(X_test, Y_test, verbose = 2, batch_size = batch_size)
print(f"Test Accuracy: {acc:.2%}")

57/57 - 1s - 20ms/step - accuracy: 0.6504 - loss: 0.8030
Test Accuracy: 65.04%


In [ ]:
from tensorflow.keras.layers import SimpleRNN, Input

# Define parameters
max_features = 5000
embed_dim = 128
rnn_units = 64 # SimpleRNN usually requires fewer units to stay stable
input_length = X.shape[1]

model_rnn = Sequential()
model_rnn.add(Input(shape=(input_length,)))
model_rnn.add(Embedding(max_features, embed_dim))
model_rnn.add(SpatialDropout1D(0.3))
model_rnn.add(SimpleRNN(rnn_units, dropout=0.2, recurrent_dropout=0.2))
model_rnn.add(Dense(4, activation='softmax')) # Classifying into four classes

model_rnn.compile(loss='categorical_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])

print(model_rnn.summary())

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 24, 128)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_2             │ (None, 24, 128)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 652,612 (2.49 MB)

 Trainable params: 652,612 (2.49 MB)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
# Train the model
history_rnn = model_rnn.fit(X_train, Y_train,
                            epochs=10,
                            batch_size=32,
                            validation_data=(X_test, Y_test),
                            callbacks=[early_stop], # Re-using the early_stop from earlier
                            verbose=2)

# Evaluate the SimpleRNN [cite: 11]
score_rnn, acc_rnn = model_rnn.evaluate(X_test, Y_test, verbose=2)
print(f"SimpleRNN Test Accuracy: {acc_rnn:.2%}")

Epoch 1/10
228/228 - 8s - 34ms/step - accuracy: 0.5558 - loss: 0.9932 - val_accuracy: 0.5899 - val_loss: 0.9086
Epoch 2/10
228/228 - 4s - 16ms/step - accuracy: 0.6227 - loss: 0.8697 - val_accuracy: 0.6003 - val_loss: 0.8895
Epoch 3/10
228/228 - 4s - 16ms/step - accuracy: 0.6972 - loss: 0.7457 - val_accuracy: 0.5976 - val_loss: 0.9068
57/57 - 0s - 8ms/step - accuracy: 0.5899 - loss: 0.9086
SimpleRNN Test Accuracy: 58.99%
